In [ ]:
# ✅ 패키지 설치 (처음 한 번만)
import subprocess
subprocess.run(['pip', 'install', 'missingno', 'xgboost', 'lightgbm', 'scikit-learn==1.5.2'])

In [ ]:
# ✅ sklearn HTML 렌더링 끄기 (에러 방지)
from sklearn import set_config
set_config(display='text')

In [ ]:
# ✅ 라이브러리 불러오기
import warnings
warnings.filterwarnings('ignore')

import os
from os.path import join

import pandas as pd
import numpy as np

import missingno as msno

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score, train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import matplotlib.pyplot as plt
import seaborn as sns

print('✅ 라이브러리 로드 완료')

In [ ]:
# ✅ 파일 경로 설정
data_dir = os.path.join(os.getenv('HOME'), 'work/aiffel_test/Data_Analysis/DA02')

train_data_path = os.path.join(data_dir, 'train.csv')
sub_data_path   = os.path.join(data_dir, 'test.csv')

# 경로 확인
print(train_data_path)
print(sub_data_path)

# 폴더 내 파일 목록 확인
print('\n📂 파일 목록:')
print(os.listdir(data_dir))

In [ ]:
# ✅ 데이터 불러오기
data = pd.read_csv(train_data_path)
sub  = pd.read_csv(sub_data_path)

print(f'train data dim : {data.shape}')  # (15035, 21)
print(f'sub data dim   : {sub.shape}')   # (6468, 20)

In [ ]:
# ✅ 타겟(y) 분리
y = data['price']
del data['price']

print(data.columns)

In [ ]:
# ✅ 학습/테스트 데이터 합치기
train_len = len(data)                   # 나중에 분리할 기준점 (15035)
data = pd.concat((data, sub), axis=0)   # 위아래로 합치기

print(f'합친 데이터 크기: {len(data)}')  # 21503

In [ ]:
# ✅ 결측치 시각화
msno.matrix(data)
plt.show()

In [ ]:
# ✅ 모든 컬럼 결측치 개수 확인
for c in data.columns:
    print('{} : {}'.format(c, len(data.loc[pd.isnull(data[c]), c].values)))

In [ ]:
# ✅ id 컬럼 정리
sub_id = data['id'][train_len:]   # 제출용 id 따로 저장
del data['id']                    # id 삭제

print(data.columns)

In [ ]:
# ✅ date 컬럼 처리: 앞 6자리(YYYYMM)만 추출
data['date'] = data['date'].apply(lambda x: str(x[:6]))

data.head()

In [ ]:
# ✅ 학습(x) / 테스트(sub) 재분리
sub = data.iloc[train_len:, :]   # 테스트 데이터
x   = data.iloc[:train_len, :]   # 학습 데이터

print(f'x shape   : {x.shape}')    # (15035, 19)
print(f'sub shape : {sub.shape}')  # (6468, 19)

In [ ]:
# ✅ 타겟 분포 확인 (오른쪽으로 치우친 분포)
sns.kdeplot(y)
plt.title('y 분포 (원본)')
plt.show()

In [ ]:
# ✅ log 변환 (분포를 정규분포에 가깝게)
y_log = np.log1p(y)

sns.kdeplot(y_log)
plt.title('y 분포 (log 변환 후)')
plt.show()

In [ ]:
# ✅ RMSE 함수 정의
def rmse(y_test, y_pred):
    # log 변환된 값을 역변환 후 RMSE 계산
    return np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred)))

In [ ]:
# ✅ 모델 초기화
random_state = 2020

gboost   = GradientBoostingRegressor(random_state=random_state)
xgboost  = XGBRegressor(random_state=random_state)
lightgbm = LGBMRegressor(random_state=random_state)
rdforest = RandomForestRegressor(random_state=random_state)

models = [gboost, xgboost, lightgbm, rdforest]

print('✅ 모델 초기화 완료')

In [ ]:
# ✅ 각 모델 RMSE 평가
df = {}

for model in models:
    model_name = model.__class__.__name__

    X_train, X_test, y_train, y_test = train_test_split(
        x, y_log, random_state=random_state, test_size=0.2)  # ✅ y_log 사용

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    df[model_name] = rmse(y_test, y_pred)
    score_df = pd.DataFrame(df, index=['RMSE']).T.sort_values('RMSE', ascending=False)

score_df

In [ ]:
# ✅ get_scores 함수 정의
def get_scores(models, x, y):
    df = {}
    for model in models:
        model_name = model.__class__.__name__

        X_train, X_test, y_train, y_test = train_test_split(
            x, y, random_state=random_state, test_size=0.2)

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        df[model_name] = rmse(y_test, y_pred)
        score_df = pd.DataFrame(df, index=['RMSE']).T.sort_values('RMSE', ascending=False)

    return score_df

In [ ]:
# ✅ my_GridSearch 함수 정의
def my_GridSearch(model, train, y, param_grid, verbose=2, n_jobs=5):
    grid_model = GridSearchCV(model, param_grid=param_grid,
                              scoring='neg_mean_squared_error',
                              cv=5, verbose=verbose, n_jobs=n_jobs)
    grid_model.fit(train, y)  # ✅ y_log 전달받아 학습

    params  = grid_model.cv_results_['params']
    score   = grid_model.cv_results_['mean_test_score']

    results = pd.DataFrame(params)
    results['score'] = score
    results['RMSLE'] = np.sqrt(-1 * results['score'])
    results = results.sort_values('RMSLE')

    return results

In [ ]:
# ✅ GridSearch 파라미터 탐색 (확대된 범위)
param_grid = {
    'n_estimators':  [500, 1000, 2000],   # ✅ 기존 [50,100] → 확대
    'max_depth':     [5, 7, 10],           # ✅ 기존 [1,10]  → 세분화
    'learning_rate': [0.01, 0.05, 0.1],   # ✅ 새로 추가
    'num_leaves':    [31, 63, 127],        # ✅ 새로 추가
}

model = LGBMRegressor(random_state=random_state)
best_result = my_GridSearch(model, x, y_log, param_grid, verbose=1, n_jobs=-1)  # ✅ y_log 사용
best_result.head()

In [ ]:
# ✅ 최적 파라미터로 최종 모델 학습 및 예측
best_params = best_result.iloc[0]   # RMSLE 가장 낮은 파라미터 추출
print('최적 파라미터:', best_params)

final_model = LGBMRegressor(
    max_depth     = int(best_params['max_depth']),
    n_estimators  = int(best_params['n_estimators']),
    learning_rate = best_params['learning_rate'],
    num_leaves    = int(best_params['num_leaves']),
    random_state  = random_state,
    verbose       = -1
)

final_model.fit(x, y_log)              # ✅ y_log로 학습
prediction = final_model.predict(sub)  # 테스트 데이터 예측
prediction = np.expm1(prediction)      # ✅ 역변환 (log → 실제 가격)

print(f'예측 개수: {len(prediction)}')
prediction

In [ ]:
# ✅ 제출 파일 생성
result = pd.DataFrame({
    'id'    : sub_id,
    'price' : prediction
})

result.head()

In [ ]:
# ✅ CSV 파일로 저장
my_submission_path = join(data_dir, 'submission.csv')
result.to_csv(my_submission_path, index=False)

print(f'✅ 저장 완료: {my_submission_path}')